# Fragrance Success Prediction – SHAP Analysis

This notebook explains the predictions of the trained **full Random Forest model** using SHAP.

## Goals

- identify the most important features driving predicted fragrance success
- explain individual perfume predictions
- generate plots and saved outputs for reporting

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings("ignore")

print("shap version:", shap.__version__)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..")

DATA_PATH    = PROJECT_ROOT / "data"    / "processed" / "fragrantica_features.parquet"
MODELS_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR  = PROJECT_ROOT / "reports" / "results"
FIGURES_DIR  = PROJECT_ROOT / "reports" / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_PATH   = RESULTS_DIR / "modeling_artifacts.json"
FULL_MODEL_PATH  = MODELS_DIR  / "rf_full.pkl"
CONFIG_PATH      = MODELS_DIR  / "model_config.json"

print("DATA_PATH:       ", DATA_PATH)
print("FULL_MODEL_PATH: ", FULL_MODEL_PATH)
print("CONFIG_PATH:     ", CONFIG_PATH)
print("RESULTS_DIR:     ", RESULTS_DIR)
print("FIGURES_DIR:     ", FIGURES_DIR)

In [ ]:
df    = pd.read_parquet(DATA_PATH)
model = joblib.load(FULL_MODEL_PATH)

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r") as f:
        model_config = json.load(f)
else:
    model_config = {}

tuned_threshold = model_config.get("rf_full_threshold", 0.5)

print("Data shape:      ", df.shape)
print("Loaded model:    ", type(model).__name__)
print("Tuned threshold: ", tuned_threshold)

In [ ]:
# Build SHAP feature matrix using the exact model input features

if not ARTIFACTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing artifacts file: {ARTIFACTS_PATH}. "
        "Run the export-artifacts cell in 04_modeling.ipynb first."
    )

with open(ARTIFACTS_PATH, "r", encoding="utf-8") as f:
    artifacts = json.load(f)

model_features = artifacts["full_cols"]

missing_features = [col for col in model_features if col not in df.columns]
if missing_features:
    raise ValueError(
        f"These trained features are missing from the parquet file: {missing_features[:20]}"
    )

X_shap = df[model_features].astype(float).copy()

print("X_shap shape:          ", X_shap.shape)
print("Number of model features:", len(model_features))
X_shap.head()

In [ ]:
# Keep perfume metadata for local explanations

meta_cols = [col for col in ["perfume", "brand", "year", "gender", "country"] if col in df.columns]
meta_df   = df[meta_cols].copy()

print("Metadata columns kept:", meta_cols)
meta_df.head()

In [ ]:
# Create smaller background and explanation samples for speed

RANDOM_STATE = 42

background_size = min(200,  len(X_shap))
explain_size    = min(1000, len(X_shap))

background_idx = X_shap.sample(n=background_size, random_state=RANDOM_STATE).index
explain_idx    = X_shap.sample(n=explain_size,    random_state=RANDOM_STATE).index

X_background = X_shap.loc[background_idx]
X_explain    = X_shap.loc[explain_idx]

print("Background sample shape:", X_background.shape)
print("Explain sample shape:   ", X_explain.shape)

In [ ]:
# Build SHAP explainer and compute SHAP values

explainer       = shap.TreeExplainer(model, data=X_background)
shap_values_raw = explainer.shap_values(X_explain)

# Normalise to a 2-D array of shape (n_samples, n_features) for the positive class.
# SHAP >= 0.46 returns a 3-D ndarray (n_samples, n_features, n_classes);
# older versions return a list-of-arrays [class0, class1].
shap_arr = np.array(shap_values_raw)

if shap_arr.ndim == 3:
    # New-style: shape (n_samples, n_features, n_classes)
    shap_values  = shap_arr[:, :, 1]
    ev           = explainer.expected_value
    expected_value = float(np.atleast_1d(ev)[1]) if np.atleast_1d(ev).shape[0] > 1 else float(ev)
elif isinstance(shap_values_raw, list):
    # Old-style: list of two arrays
    shap_values  = np.array(shap_values_raw[1])
    ev           = explainer.expected_value
    expected_value = float(np.atleast_1d(ev)[1]) if np.atleast_1d(ev).shape[0] > 1 else float(ev)
else:
    shap_values  = shap_arr
    ev           = explainer.expected_value
    expected_value = float(np.atleast_1d(ev)[1]) if np.atleast_1d(ev).shape[0] > 1 else float(ev)

print("SHAP values shape:", shap_values.shape)
print("Expected value:   ", expected_value)

In [ ]:
# Global SHAP summary plot (beeswarm)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_explain, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_beeswarm_full_model.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Global SHAP bar plot

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_explain, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_bar_full_model.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP importance table

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance_df = (
    pd.DataFrame({"feature": X_explain.columns, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_df.to_csv(
    RESULTS_DIR / "shap_feature_importance_full_model.csv",
    index=False
)

print("Top 20 SHAP features:")
display(shap_importance_df.head(20))

In [ ]:
# Save top-20 SHAP features

top_20_shap = shap_importance_df.head(20).copy()
top_20_shap.to_csv(
    RESULTS_DIR / "shap_top20_features_full_model.csv",
    index=False
)

top_20_shap

In [ ]:
# Model prediction probabilities on the explanation sample

pred_proba         = model.predict_proba(X_explain)[:, 1]
pred_label_default = (pred_proba >= 0.5).astype(int)
pred_label_tuned   = (pred_proba >= tuned_threshold).astype(int)

prediction_df = pd.DataFrame({
    "index":                X_explain.index,
    "pred_proba_success":   pred_proba,
    "pred_label_default_0_5": pred_label_default,
    "pred_label_tuned":     pred_label_tuned,
})

if "perfume" in meta_df.columns:
    prediction_df["perfume"] = meta_df.loc[X_explain.index, "perfume"].values
if "brand" in meta_df.columns:
    prediction_df["brand"]   = meta_df.loc[X_explain.index, "brand"].values

prediction_df.to_csv(
    RESULTS_DIR / "shap_prediction_sample_full_model.csv",
    index=False
)

prediction_df.head()

In [ ]:
# Build a SHAP Explanation object for waterfall / force plots

shap_explanation = shap.Explanation(
    values        = shap_values,
    base_values   = np.repeat(expected_value, len(X_explain)),
    data          = X_explain.values,
    feature_names = list(X_explain.columns)
)

print("SHAP Explanation object created.")
print("values shape:     ", shap_explanation.values.shape)
print("base_values shape:", shap_explanation.base_values.shape)

In [ ]:
# Inspect the explanation sample — use this to choose a perfume name for the next cell

sample_lookup = meta_df.loc[X_explain.index].copy()
sample_lookup["pred_proba_success"] = pred_proba
sample_lookup["pred_label_tuned"]   = pred_label_tuned

sample_lookup.head(20)

In [ ]:
# Function: explain a perfume by name (waterfall plot + local contribution table)

def explain_perfume_by_name(perfume_name, save_figure=True):
    if "perfume" not in meta_df.columns:
        raise ValueError("Column 'perfume' not found in metadata.")

    matches      = meta_df.loc[X_explain.index]
    matched_rows = matches[matches["perfume"].astype(str).str.lower() == perfume_name.lower()]

    if matched_rows.empty:
        print(f"No perfume named '{perfume_name}' found in the explanation sample.")
        print("Tip: check sample_lookup above for available names.")
        return None

    row_idx = matched_rows.index[0]
    pos     = list(X_explain.index).index(row_idx)

    print("Perfume:",  meta_df.loc[row_idx, "perfume"] if "perfume" in meta_df.columns else "N/A")
    print("Brand:  ",  meta_df.loc[row_idx, "brand"]   if "brand"   in meta_df.columns else "N/A")
    print("Predicted probability of success:", round(float(pred_proba[pos]), 4))
    print("Predicted class (tuned threshold):", int(pred_label_tuned[pos]))

    plt.figure(figsize=(12, 6))
    shap.plots.waterfall(shap_explanation[pos], max_display=15, show=False)
    plt.tight_layout()

    if save_figure:
        safe_name = "".join(
            c if c.isalnum() or c in ("_", "-") else "_"
            for c in perfume_name.lower().replace(" ", "_")
        )
        plt.savefig(FIGURES_DIR / f"waterfall_{safe_name}.png", dpi=300, bbox_inches="tight")

    plt.show()

    local_contrib = (
        pd.DataFrame({
            "feature":       X_explain.columns,
            "feature_value": X_explain.iloc[pos].values,
            "shap_value":    shap_values[pos],
        })
        .sort_values("shap_value", ascending=False)
        .reset_index(drop=True)
    )

    return local_contrib

print("explain_perfume_by_name() defined.")

## Example: explain a specific perfume

First inspect `sample_lookup` (cell above) to find a perfume name that is in the sample.  
Then replace `"Sauvage"` below with any name from that list.

In [ ]:
# Available names in the explanation sample
sample_lookup[["perfume", "brand", "pred_proba_success"]].sort_values(
    "pred_proba_success", ascending=False
).head(20)

In [ ]:
# Replace with a name visible above
local_contrib_df = explain_perfume_by_name("Sauvage")

if local_contrib_df is not None:
    display(local_contrib_df.head(15))

In [ ]:
# Dependence plots for the top-3 SHAP features

top_features = shap_importance_df["feature"].head(3).tolist()
print("Top features for dependence plots:", top_features)

for feature_name in top_features:
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(feature_name, shap_values, X_explain, show=False)
    plt.tight_layout()
    safe_feature = feature_name.replace("/", "_").replace(" ", "_")
    plt.savefig(FIGURES_DIR / f"shap_dependence_{safe_feature}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Compact interpretation table for report writing (top 15)

report_table = shap_importance_df.head(15).copy()
report_table["rank"] = range(1, len(report_table) + 1)
report_table = report_table[["rank", "feature", "mean_abs_shap"]]

report_table.to_csv(
    RESULTS_DIR / "shap_report_table_top15.csv",
    index=False
)

report_table

In [ ]:
# Summary of saved files

saved = [
    FIGURES_DIR  / "shap_summary_beeswarm_full_model.png",
    FIGURES_DIR  / "shap_summary_bar_full_model.png",
    RESULTS_DIR  / "shap_feature_importance_full_model.csv",
    RESULTS_DIR  / "shap_top20_features_full_model.csv",
    RESULTS_DIR  / "shap_prediction_sample_full_model.csv",
    RESULTS_DIR  / "shap_report_table_top15.csv",
]

print("Saved files:")
for p in saved:
    status = "✓" if p.exists() else "✗ (not yet written)"
    print(f"  {status}  {p}")

print("\nSHAP analysis complete.")